In [ ]:
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.compose import TransformedTargetRegressor, ColumnTransformer, make_column_selector
from sklearn.feature_selection import VarianceThreshold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
transition_model = pd.read_parquet("phenocam_phenology_model.parquet")

In [ ]:
class VonMisesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cycle_length=365):
        self.cycle_length = cycle_length
        self.to_rad_ = (2 * np.pi) / cycle_length
        self.mean_angle_ = None

    def to_radians(self, x):
        return x * self.to_rad_

    def from_radians(self, x):
        return x / self.to_rad_
        
    def fit(self, X, y=None):
        self.mean_angle_ = np.mean(self.to_radians(X))
        return self
        
    def transform(self, X, y=None):
        return np.tan((self.to_radians(X) - self.mean_angle_) / 2)

    def inverse_transform(self, X, y=None):
        return self.from_radians(2 * np.atan(X) + self.mean_angle_)
        

In [ ]:
vmt = VonMisesTransformer(cycle_length=365)
vmt.fit(transition_model["doy_transition_50_rising"])

plt.scatter(transition_model["doy_transition_50_rising"], vmt.inverse_transform(vmt.transform(transition_model["doy_transition_50_rising"])))

In [ ]:
def angular_transform(x, cycle_length=365):
    x_rad = (x * (2*np.pi) / cycle_length) % (2 * np.pi) # Convert to radians
    return np.stack([np.cos(x_rad), np.sin(x_rad)], axis=1)

def inverse_angular_transform(x_cos, x_sin, cycle_length=365):
    return np.atan2(x_sin, x_cos) * cycle_length / (2 * np.pi)

transformed = inverse_angular_transform(*angular_transform(transition_model["doy_transition_50_rising"]).T)

plt.scatter(inverse_angular_transform(*angular_transform(transition_model["doy_transition_50_rising"]).T), transition_model["doy_transition_50_rising"])

In [ ]:
doy_to_rad = (360 / 365) * (np.pi / 180)

def von_mises_link(x):
    x_rad = x * doy_to_rad
    return 2 * np.tan(x_rad / 2)

def von_mises_inverse(x):
    x_rad = 2 * np.atan(x / 2)
    x_doy = x_rad * (1 / doy_to_rad)
    return x_doy

In [ ]:
np.atan(

In [ ]:
plt.scatter(transition_model["tmax_lt"], transition_model["doy_transition_50_rising"])

In [ ]:
plt.scatter(transition_model["tmax_lt"], von_mises_link(transition_model["doy_transition_50_rising"]))
plt.ylim(-20, 20)

In [ ]:
transition_model.columns

In [ ]:
input_features = ["tmax_lt", "prcp_lt", "tmax_anomaly", "prcp_anomaly", "veg_type"]
target = "doy_transition_50_rising"

y = transition_model[target]
X = transition_model[input_features]

ct = ColumnTransformer(
    transformers=[
        (
            'category', 
            OneHotEncoder(sparse_output=False), 
            make_column_selector(dtype_include=['object', 'category'])
        ),
        (
            'scaler',
            StandardScaler(),
            make_column_selector(dtype_include=["float"])
        )
    ],
    remainder='passthrough'  # Keeps others untouched
)

ct.fit(X)

poly = PolynomialFeatures(2, interaction_only=True)
poly.fit(ct.transform(X))

poly_feature_names = poly.get_feature_names_out(ct.get_feature_names_out())

# Only retain features that are first-order numerics or
# interactions with one of the numerics. Numeric variables
# will be prepended with "scaler", so we can use this with a regex
# to figure out what columns to keep.
features_to_retain = [i for i, x in enumerate(poly_features_names) if x == '1' or x.count("scaler") == 1]

col_filter = ColumnTransformer(
    transformers=[
        ('filter', "passthrough", features_to_retain)
    ],
    remainder="drop"
)

In [ ]:
regressor = TransformedTargetRegressor(
    LinearRegression(),
    transformer=VonMisesTransformer(cycle_length=365)
)

pipe = Pipeline([
    ("preprocess", ct),
    ("interactions", poly),
    ("feat_selection", col_filter),
    ("regressor", regressor),
    
])

# This transformation has to be done outside of the pipeline
# bc sklearn does not support stateless transformations on
# y only.
y_tform = angular_transform(y)

pipe.fit(X, y)

In [ ]:
X_copy = X.assign(
    doy_true = y,
    doy_predict = pipe.predict(X)
)

fig, axes = plt.subplots(2, 3, sharex=True, sharey=True)

for(veg, df), ax in zip(X_copy.groupby("veg_type"), axes.flat):
    ax.scatter(df["doy_true"], df["doy_predict"], s=1)
    ax.set_title(veg)
    ax.axline((0, 0), (1, 1), color="black")
    ax.grid()
    ax.set_aspect("equal")

fig.supxlabel("True rising DOY")
fig.supylabel("Model rising DOY")
plt.show()

In [ ]:
plt.hist(2 * np.tan((y * np.pi / 180) / 2))

In [ ]:
np.tan(plt.hist(y * np.pi / 180)

In [ ]:
def jammalamadaka_sharma_corr(alpha, beta):
    """
    Computes the Jammalamadaka-Sharma circular correlation coefficient.
    Inputs must be array-like and in radians.
    """
    alpha = np.asarray(alpha)
    beta = np.asarray(beta)
    
    # Compute circular means using mean of sines and cosines
    mean_alpha = np.arctan2(np.mean(np.sin(alpha)), np.mean(np.cos(alpha)))
    mean_beta = np.arctan2(np.mean(np.sin(beta)), np.mean(np.cos(beta)))
    
    # Compute deviations
    sin_alpha_dev = np.sin(alpha - mean_alpha)
    sin_beta_dev = np.sin(beta - mean_beta)
    
    # Calculate numerator and denominator components
    num = np.sum(sin_alpha_dev * sin_beta_dev)
    den = np.sqrt(np.sum(sin_alpha_dev**2) * np.sum(sin_beta_dev**2))
    
    if den == 0:
        return 0.0
    return num / den

In [ ]:
jammalamadaka_sharma_corr((y * np.pi / 180), inverse_angular_transform(*pipe.predict(X).T, cycle_length=2*np.pi))

In [ ]:
regressor = MultiOutputRegressor(LinearRegression())

y = transition_model["doy_transition_50_rising"]
X = transition_model[["tmax_lt", "prcp_lt", "veg_type"]]

y_tform = angular_transform(y)

regressor.fit(X, y_tform)

In [ ]:
plt.scatter(y, inverse_angular_transform(*regressor.predict(X).T))

In [ ]:
regressor.fit(transition_model[["tmax_lt", "prcp_lt"]], transition_model["doy_transition_50_rising"], check_inverse=False)

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import PowerTransformer

# 1. Create a dummy multi-output dataset
X, y = make_regression(n_samples=100, n_features=5, n_targets=3, random_state=42)

# Ensure targets are positive to simulate a scenario needing transformation
y = np.abs(y) + 1.0 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define the base estimator (does not need to natively support multi-output)
base_estimator = Ridge()

# 3. Wrap it in MultiOutputRegressor to handle multiple targets
multi_regressor = MultiOutputRegressor(estimator=base_estimator)

# 4. Wrap everything inside TransformedTargetRegressor
# This transforms y before passing it to the MultiOutputRegressor, 
# and automatically maps predictions back to the original scale.
model = TransformedTargetRegressor(
    regressor=multi_regressor,
    transformer=PowerTransformer(method='box-cox')
)

# 5. Fit and Predict
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Predictions Shape:", predictions.shape)  # Output: (20, 3)


In [ ]:
transition_model["doy_transition_50_rising"].hist()

In [ ]:
(-200 * 2 * np.pi / 365) % (2*np.pi)